Load dataset

In [ ]:
from workflow import tracks_import as importD
import glob
import os
from pprint import pprint
import pandas as pd
import numpy as np
import seaborn as sns

base_path = os.path.join('data')
print(f'{base_path=}')
print("Absolute path:", os.path.abspath(base_path))

In [ ]:
import os

# =========================
# Find all files
# =========================
tracks_files = sorted(glob.glob(os.path.join(base_path, "*_tracks.csv")))
tracks_meta_files = sorted(glob.glob(os.path.join(base_path, "*_tracksMeta.csv")))
recording_meta_files = sorted(glob.glob(os.path.join(base_path, "*_recordingMeta.csv")))

print(f"Found {len(tracks_files)} tracks files")
print(f"Found {len(tracks_meta_files)} tracksMeta files")
print(f"Found {len(recording_meta_files)} recordingMeta files")

if len(recording_meta_files) == 0:
    raise FileNotFoundError("No recordingMeta files found. Please check base_path.")

Merging vehicle extraction for all files

In [ ]:
# =========================
# Merging lanelet ID mapping
# =========================
def get_merging_lanelet_id(recording):
    if 0 <= recording <= 18:
        return 1754
    elif 19 <= recording <= 38:
        return 1966
    elif 39 <= recording <= 52:
        return 1494
    elif 53 <= recording <= 60:
        return 1408
    elif 61 <= recording <= 72:
        return 1469
    elif 73 <= recording <= 77:
        return 1405
    elif 78 <= recording <= 92:
        return 1455
    else:
        raise ValueError(f"Recording {recording} is outside the known range.")


# =========================
# Extract merging vehicles
# =========================
def extract_merging_vehicles(tracks_df, merging_lanelet_id):
    merging_ids = []

    for _, row in tracks_df.iterrows():
        track_id = row["trackId"]
        lanelets = np.asarray(row["laneletId"])

        lanelets = lanelets[~pd.isna(lanelets)]
        unique_lanelets = np.unique(lanelets)

        if merging_lanelet_id in unique_lanelets and len(unique_lanelets) > 1:
            merging_ids.append(int(track_id))

    return merging_ids


# =========================
# Loop over all recordings
# =========================
all_merging_vehicles = []
summary_results = []

for recording in range(93):
    track_file = os.path.join(base_path, f"{recording:02d}_tracks.csv")
    tracks_meta_file = os.path.join(base_path, f"{recording:02d}_tracksMeta.csv")
    recording_meta_file = os.path.join(base_path, f"{recording:02d}_recordingMeta.csv")

    if not (
        os.path.exists(track_file)
        and os.path.exists(tracks_meta_file)
        and os.path.exists(recording_meta_file)
    ):
        print(f"Recording {recording:02d} missing files, skipping")
        continue

    tracks, tracks_meta, recording_meta = importD.read_from_csv(
        track_file,
        tracks_meta_file,
        recording_meta_file
    )

    tracks_df = pd.DataFrame(tracks)

    merging_lanelet_id = get_merging_lanelet_id(recording)

    merging_ids = extract_merging_vehicles(
        tracks_df=tracks_df,
        merging_lanelet_id=merging_lanelet_id
    )

    print(f"\nRecording {recording:02d}")
    print(f"Merging lanelet ID: {merging_lanelet_id}")
    print(f"Number of merging vehicles: {len(merging_ids)}")
    print(f"Merging vehicle IDs: {merging_ids}")

    summary_results.append({
        "recording_id": recording,
        "merging_lanelet_id": merging_lanelet_id,
        "num_merging_vehicles": len(merging_ids),
        "merging_vehicle_ids": merging_ids
    })

    for track_id in merging_ids:
        all_merging_vehicles.append({
            "recording_id": recording,
            "track_id": track_id,
            "merging_lanelet_id": merging_lanelet_id
        })


# =========================
# Save recording-level summary
# =========================
summary_df = pd.DataFrame(summary_results)

display(summary_df.head())

os.makedirs(os.path.join("results", "summary"), exist_ok=True)
summary_df.to_csv(
    os.path.join("results", "summary", "exid_merging_vehicle_summary.csv"),
    index=False
)

print("Saved: results/summary/exid_merging_vehicle_summary.csv")